In [1]:
import re
import hashlib
from datetime import datetime
import numpy as np
import string

# Parse the data
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords
from nltk.stem.wordnet import WordNetLemmatizer
import string

In [40]:
from pyspark.sql import SparkSession, Row
from pyspark.ml.feature import CountVectorizer, Tokenizer, StopWordsRemover, HashingTF, IDF, StringIndexer, VectorAssembler
from pyspark.ml.clustering import LDA
from pyspark.ml.linalg import SparseVector
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.sql.functions import udf, col, concat_ws, length, split
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc as sk_auc, confusion_matrix, ConfusionMatrixDisplay

In [41]:
spark = SparkSession.builder.appName("News_Recommendation_using_EnsembleMethod").getOrCreate()

In [16]:
from pyspark.sql.types import StructType, StructField, StringType

news_schema = StructType([
    StructField("News ID", StringType(), True),
    StructField("Category", StringType(), True),
    StructField("SubCategory", StringType(), True),
    StructField("Title", StringType(), True),
    StructField("Abstract", StringType(), True),
    StructField("URL", StringType(), True),
    StructField("Title Entities", StringType(), True),
    StructField("Abstract Entities", StringType(), True)
])

news_df = (
    spark.read
    .option("header", False)
    .option("sep", "\t") 
    .option("quote", '"')
    .option("escape", '"')
    .option("multiLine", True)
    .schema(news_schema)
    .csv("gs://housespark-bucket1/news.tsv")
)
news_df.printSchema()
news_df.show(3, truncate=True)

root
 |-- News ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- SubCategory: string (nullable = true)
 |-- Title: string (nullable = true)
 |-- Abstract: string (nullable = true)
 |-- URL: string (nullable = true)
 |-- Title Entities: string (nullable = true)
 |-- Abstract Entities: string (nullable = true)

+-------+---------+---------------+--------------------+--------------------+--------------------+--------------------+--------------------+
|News ID| Category|    SubCategory|               Title|            Abstract|                 URL|      Title Entities|   Abstract Entities|
+-------+---------+---------------+--------------------+--------------------+--------------------+--------------------+--------------------+
| N55528|lifestyle|lifestyleroyals|The Brands Queen ...|Shop the notebook...|https://assets.ms...|[{"Label": "Princ...|                  []|
| N19639|   health|     weightloss|50 Worst Habits F...|These seemingly h...|https://assets.ms...|[{"

In [17]:
news_df = news_df.drop("Title Entities", "Abstract Entities")
news_df.printSchema()
news_df.show(5)

root
 |-- News ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- SubCategory: string (nullable = true)
 |-- Title: string (nullable = true)
 |-- Abstract: string (nullable = true)
 |-- URL: string (nullable = true)

+-------+---------+---------------+--------------------+--------------------+--------------------+
|News ID| Category|    SubCategory|               Title|            Abstract|                 URL|
+-------+---------+---------------+--------------------+--------------------+--------------------+
| N55528|lifestyle|lifestyleroyals|The Brands Queen ...|Shop the notebook...|https://assets.ms...|
| N19639|   health|     weightloss|50 Worst Habits F...|These seemingly h...|https://assets.ms...|
| N61837|     news|      newsworld|The Cost of Trump...|Lt. Ivan Molchane...|https://assets.ms...|
| N53526|   health|         voices|I Was An NBA Wife...|I felt like I was...|https://assets.ms...|
| N38324|   health|        medical|How to Get Rid of...|They seem h

In [18]:
from pyspark.sql.functions import regexp_replace

news_df = news_df.withColumn("News ID", regexp_replace("News ID", "N", ""))

In [19]:
behaviors_schema = StructType([
    StructField("Impression ID", StringType(), True),
    StructField("User ID", StringType(), True),
    StructField("Time", StringType(), True),
    StructField("History", StringType(), True),
    StructField("Impressions", StringType(), True)
])

behaviors_df = (
    spark.read
    .option("header", False)
    .option("sep", "\t")
    .option("quote", '"')
    .option("escape", '"')
    .option("multiLine", True)
    .schema(behaviors_schema)
    .csv("gs://housespark-bucket1/behaviors.tsv")
)

behaviors_df.printSchema()
behaviors_df.show(3, truncate = True)

root
 |-- Impression ID: string (nullable = true)
 |-- User ID: string (nullable = true)
 |-- Time: string (nullable = true)
 |-- History: string (nullable = true)
 |-- Impressions: string (nullable = true)

+-------------+-------+--------------------+--------------------+--------------------+
|Impression ID|User ID|                Time|             History|         Impressions|
+-------------+-------+--------------------+--------------------+--------------------+
|            1| U13740|11/11/2019 9:05:5...|N55189 N42782 N34...|   N55689-1 N35729-0|
|            2| U91836|11/12/2019 6:11:3...|N31739 N6072 N630...|N20678-0 N39317-0...|
|            3| U73700|11/14/2019 7:01:4...|N10732 N25792 N75...|N50014-0 N23877-0...|
+-------------+-------+--------------------+--------------------+--------------------+
only showing top 3 rows



In [20]:
behaviors_df = behaviors_df.withColumn("User ID", regexp_replace("User ID", "^U", ""))
behaviors_df = behaviors_df.withColumn("Impressions", regexp_replace("Impressions", "N", ""))

from pyspark.sql.functions import transform, concat_ws, split

behaviors_df = behaviors_df.withColumn("HistoryList", split("History", " "))
behaviors_df = behaviors_df.withColumn(
    "HistoryList",
    transform("HistoryList", lambda x: regexp_replace(x, "N", ""))
)

behaviors_df = behaviors_df.withColumn("History", concat_ws(" ", "HistoryList"))

In [21]:
from pyspark.sql.functions import split

behaviors_df_0 = behaviors_df.withColumn("ImpressionList", split("Impressions", " "))
behaviors_df_0.select("ImpressionList").show(1, False)

+------------------+
|ImpressionList    |
+------------------+
|[55689-1, 35729-0]|
+------------------+
only showing top 1 row



In [22]:
from pyspark.sql.functions import explode, split, col

behaviors_df_1 = behaviors_df_0.select(
    "User ID", "Time", "History",
    explode(split("Impressions", " ")).alias("News_Impression")
)

behaviors_df_1 = behaviors_df_1.withColumn("News_ID", split(col("News_Impression"), "-")[0]) \
                         .withColumn("Label", split(col("News_Impression"), "-")[1]) \
                         .drop("News_Impression")

behaviors_df_1.show(5)

+-------+--------------------+--------------------+-------+-----+
|User ID|                Time|             History|News_ID|Label|
+-------+--------------------+--------------------+-------+-----+
|  13740|11/11/2019 9:05:5...|55189 42782 34694...|  55689|    1|
|  13740|11/11/2019 9:05:5...|55189 42782 34694...|  35729|    0|
|  91836|11/12/2019 6:11:3...|31739 6072 63045 ...|  20678|    0|
|  91836|11/12/2019 6:11:3...|31739 6072 63045 ...|  39317|    0|
|  91836|11/12/2019 6:11:3...|31739 6072 63045 ...|  58114|    0|
+-------+--------------------+--------------------+-------+-----+
only showing top 5 rows



In [23]:
from pyspark.sql.functions import split, explode

history_df = behaviors_df.select("User ID", "History")

history_df = history_df.withColumn("HistoryList", split("History", " ")) \
                       .select("User ID", explode("HistoryList").alias("History_News_ID"))

history_df.show(10)

+-------+---------------+
|User ID|History_News_ID|
+-------+---------------+
|  13740|          55189|
|  13740|          42782|
|  13740|          34694|
|  13740|          45794|
|  13740|          18445|
|  13740|          63302|
|  13740|          10414|
|  13740|          19347|
|  13740|          31801|
|  91836|          31739|
+-------+---------------+
only showing top 10 rows



In [24]:
from pyspark.sql.functions import expr

behaviors_df_2 = behaviors_df_0.withColumn(
    "ClickedNews",
    expr("""filter(ImpressionList, x -> split(x, '-')[1] = '1')""")
)

behaviors_df_2 = behaviors_df_2.withColumn(
    "UnclickedNews",
    expr("""filter(ImpressionList, x -> split(x, '-')[1] = '0')""")
)

behaviors_df_2 = behaviors_df_2.withColumn(
    "ClickedNews", expr("""transform(ClickedNews, x -> split(x, '-')[0])""")
).withColumn(
    "UnclickedNews", expr("""transform(UnclickedNews, x -> split(x, '-')[0])""")
)

behaviors_df_2.show(3)

+-------------+-------+--------------------+--------------------+--------------------+--------------------+--------------------+-----------+--------------------+
|Impression ID|User ID|                Time|             History|         Impressions|         HistoryList|      ImpressionList|ClickedNews|       UnclickedNews|
+-------------+-------+--------------------+--------------------+--------------------+--------------------+--------------------+-----------+--------------------+
|            1|  13740|11/11/2019 9:05:5...|55189 42782 34694...|     55689-1 35729-0|[55189, 42782, 34...|  [55689-1, 35729-0]|    [55689]|             [35729]|
|            2|  91836|11/12/2019 6:11:3...|31739 6072 63045 ...|20678-0 39317-0 5...|[31739, 6072, 630...|[20678-0, 39317-0...|    [17059]|[20678, 39317, 58...|
|            3|  73700|11/14/2019 7:01:4...|10732 25792 7563 ...|50014-0 23877-0 3...|[10732, 25792, 75...|[50014-0, 23877-0...|    [23814]|[50014, 23877, 35...|
+-------------+-------+-----

In [25]:
# Cast "User ID" column to IntegerType
from pyspark.sql.functions import col

behaviors_df_2 = behaviors_df_2.withColumn("User ID", col("User ID").cast("int"))
behaviors_df_2.show(3)

+-------------+-------+--------------------+--------------------+--------------------+--------------------+--------------------+-----------+--------------------+
|Impression ID|User ID|                Time|             History|         Impressions|         HistoryList|      ImpressionList|ClickedNews|       UnclickedNews|
+-------------+-------+--------------------+--------------------+--------------------+--------------------+--------------------+-----------+--------------------+
|            1|  13740|11/11/2019 9:05:5...|55189 42782 34694...|     55689-1 35729-0|[55189, 42782, 34...|  [55689-1, 35729-0]|    [55689]|             [35729]|
|            2|  91836|11/12/2019 6:11:3...|31739 6072 63045 ...|20678-0 39317-0 5...|[31739, 6072, 630...|[20678-0, 39317-0...|    [17059]|[20678, 39317, 58...|
|            3|  73700|11/14/2019 7:01:4...|10732 25792 7563 ...|50014-0 23877-0 3...|[10732, 25792, 75...|[50014-0, 23877-0...|    [23814]|[50014, 23877, 35...|
+-------------+-------+-----

### Validation Data Preparation

In [56]:
behaviors_schema = StructType([
    StructField("Impression ID", StringType(), True),
    StructField("User ID", StringType(), True),
    StructField("Time", StringType(), True),
    StructField("History", StringType(), True),
    StructField("Impressions", StringType(), True)
])

behaviors_dev = (
    spark.read
    .option("header", False)
    .option("sep", "\t")
    .option("quote", '"')
    .option("escape", '"')
    .option("multiLine", True)
    .schema(behaviors_schema)
    .csv("gs://housespark-bucket1/behaviors_dev.tsv")
)

behaviors_dev.printSchema()
behaviors_dev.show(3, truncate = True)

root
 |-- Impression ID: string (nullable = true)
 |-- User ID: string (nullable = true)
 |-- Time: string (nullable = true)
 |-- History: string (nullable = true)
 |-- Impressions: string (nullable = true)

+-------------+-------+--------------------+--------------------+--------------------+
|Impression ID|User ID|                Time|             History|         Impressions|
+-------------+-------+--------------------+--------------------+--------------------+
|            1| U80234|11/15/2019 12:37:...|N55189 N46039 N51...|N28682-0 N48740-0...|
|            2| U60458|11/15/2019 7:11:5...|N58715 N32109 N51...|N20036-0 N23513-1...|
|            3| U44190|11/15/2019 9:55:1...|N56253 N1150 N551...|N36779-0 N62365-0...|
+-------------+-------+--------------------+--------------------+--------------------+
only showing top 3 rows



In [57]:
from pyspark.sql.functions import transform, concat_ws, split, regexp_replace

behaviors_dev = behaviors_dev.withColumn("User ID", regexp_replace("User ID", "^U", ""))
behaviors_dev = behaviors_dev.withColumn("Impressions", regexp_replace("Impressions", "N", ""))

behaviors_dev = behaviors_dev.withColumn("HistoryList", split("History", " "))
behaviors_dev = behaviors_dev.withColumn(
    "HistoryList",
    transform("HistoryList", lambda x: regexp_replace(x, "N", ""))
)

behaviors_dev = behaviors_dev.withColumn("History", concat_ws(" ", "HistoryList"))

In [58]:
from pyspark.sql.functions import split

behaviors_dev_0 = behaviors_dev.withColumn("ImpressionList", split("Impressions", " "))
behaviors_dev_0.select("ImpressionList").show(1, False)

+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|ImpressionList                                                                                                                                                                                    |
+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|[28682-0, 48740-0, 31958-1, 34130-0, 6916-0, 5472-0, 50775-0, 24802-0, 19990-0, 33176-0, 62365-0, 5940-0, 6400-0, 58098-0, 42844-0, 49285-0, 51470-0, 53572-0, 11930-0, 21679-0, 55237-0, 29862-0]|
+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
only showing to

In [59]:
from pyspark.sql.functions import expr

behaviors_val = behaviors_dev_0.withColumn(
    "ClickedNews",
    expr("""filter(ImpressionList, x -> split(x, '-')[1] = '1')""")
)

behaviors_val = behaviors_val.withColumn(
    "UnclickedNews",
    expr("""filter(ImpressionList, x -> split(x, '-')[1] = '0')""")
)

behaviors_val = behaviors_val.withColumn(
    "ClickedNews", expr("""transform(ClickedNews, x -> split(x, '-')[0])""")
).withColumn(
    "UnclickedNews", expr("""transform(UnclickedNews, x -> split(x, '-')[0])""")
)

behaviors_val.show(3)

+-------------+-------+--------------------+--------------------+--------------------+--------------------+--------------------+-----------+--------------------+
|Impression ID|User ID|                Time|             History|         Impressions|         HistoryList|      ImpressionList|ClickedNews|       UnclickedNews|
+-------------+-------+--------------------+--------------------+--------------------+--------------------+--------------------+-----------+--------------------+
|            1|  80234|11/15/2019 12:37:...|55189 46039 51741...|28682-0 48740-0 3...|[55189, 46039, 51...|[28682-0, 48740-0...|    [31958]|[28682, 48740, 34...|
|            2|  60458|11/15/2019 7:11:5...|58715 32109 51180...|20036-0 23513-1 3...|[58715, 32109, 51...|[20036-0, 23513-1...|    [23513]|[20036, 32536, 46...|
|            3|  44190|11/15/2019 9:55:1...|56253 1150 55189 ...|36779-0 62365-0 5...|[56253, 1150, 551...|[36779-0, 62365-0...|     [5940]|[36779, 62365, 58...|
+-------------+-------+-----

### ALS

In [ ]:
from pyspark.sql.functions import col, posexplode, split, lit, explode, regexp_replace, transform, concat_ws
from pyspark.ml.recommendation import ALS

In [100]:
### Step 1: Time-decay from HistoryList
history_df = behaviors_df_2.select(
    col("User ID").alias("user"),  # no need .cast("int") anymore
    posexplode(split(col("History"), " ")).alias("pos", "item")
).withColumn("item", col("item").cast("int")) \
 .withColumn("rating", 1.0 / (col("pos") + 1))

# Only keep necessary columns
history_df = history_df.select("user", "item", "rating")

In [101]:
### Step 2: ClickedNews → rating = 1.0
clicked_df = behaviors_df_2.select(
    col("User ID").alias("user"),
    explode("ClickedNews").alias("item")
).withColumn("item", col("item").cast("int")) \
 .withColumn("rating", lit(1.0))

In [102]:
### Step 3: UnclickedNews → rating = 0.0
unclicked_df = behaviors_df_2.select(
    col("User ID").alias("user"),
    explode("UnclickedNews").alias("item")
).withColumn("item", col("item").cast("int")) \
 .withColumn("rating", lit(0.0))

In [103]:
### Step 4: Combine all sources
als_df = history_df.union(clicked_df).union(unclicked_df)

# Drop any rows with nulls
als_df = als_df.dropna(subset=["user", "item", "rating"])

In [104]:
### Step 5: Train ALS model
als = ALS(
    userCol="user",
    itemCol="item",
    ratingCol="rating",
    implicitPrefs=True,
    rank=50,
    maxIter=10,
    regParam=0.01,
    alpha=1.0,
    coldStartStrategy="drop"
)

model = als.fit(als_df)

In [105]:
user_recs = model.recommendForAllUsers(10) 

In [106]:
user_recs.show(2, truncate=False)

+----+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|user|recommendations                                                                                                                                                                                               |
+----+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|6   |[{42620, 5.301875}, {33276, 2.8287501}, {45794, 2.347429}, {11101, 2.3403068}, {46392, 2.2932854}, {33969, 2.0400672}, {60340, 2.029596}, {36530, 2.0010357}, {13138, 1.9756296}, {19495, 1.9720162}]         |
|8   |[{12254, 0.11824}, {33969, 0.10027554}, {60702, 0.09357843}, {6616, 0.08111857}, {55911, 0.08073194}, {47686, 0.07633733}, {9120, 0.070168

The order of users displayed is not guaranteed — Spark does not sort them by user ID. It’s based on internal partitioning and may vary across runs.

In [109]:
user_recs.filter(col("user") == 91836).show(truncate=False)

+-----+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|user |recommendations                                                                                                                                                                                       |
+-----+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|91836|[{14761, 8.718089}, {47020, 5.5859337}, {37509, 5.0851855}, {11101, 4.578164}, {54225, 4.4783645}, {40509, 4.0389047}, {41375, 3.8894026}, {13231, 3.7971928}, {39235, 3.6893785}, {56514, 3.5639076}]|
+-----+---------------------------------------------------------------------------------------------------------------------------------------------------------------------

### Content-based filtering: CBF 

In [74]:
from pyspark.ml.feature import Tokenizer, HashingTF, IDF, StringIndexer, VectorAssembler
from pyspark.ml import Pipeline

tokenizer = Tokenizer(inputCol = "Title", outputCol = "tokens")
hashingTF = HashingTF(inputCol = "tokens", outputCol = "tf", numFeatures = 100)
idf = IDF(inputCol = "tf", outputCol = "tfidf")
indexer = StringIndexer(inputCol = "Category", outputCol = "CategoryIndex")
assembler = VectorAssembler(inputCols = ["tfidf", "CategoryIndex"], outputCol = "content_features")

pipeline = Pipeline(stages = [tokenizer, hashingTF, idf, indexer, assembler])
news_model = pipeline.fit(news_df)
news_features_df = news_model.transform(news_df).select("News ID", "content_features")

In [75]:
from pyspark.ml.functions import vector_to_array
from pyspark.sql.functions import col, avg

vec_len = 100
news_features_df = news_features_df.withColumn("content_array", vector_to_array("content_features"))
for i in range(vec_len):
    news_features_df = news_features_df.withColumn(f"feature_{i}", col("content_array")[i])

In [76]:
clicked_exploded = behaviors_df_2.select("User ID", expr("explode(ClickedNews)").alias("News ID")).dropna()
user_clicks_feat = clicked_exploded.join(news_features_df, on = "News ID", how = "inner")
user_profiles = user_clicks_feat.groupBy("User ID").agg(*[avg(f"feature_{i}").alias(f"user_feat_{i}") for i in range(vec_len)])

In [77]:
from pyspark.sql.functions import sqrt, row_number
from functools import reduce
from pyspark.sql.window import Window

user_news = user_profiles.crossJoin(news_features_df.select("News ID", *[f"feature_{i}" for i in range(vec_len)]))
dot_product = reduce(lambda x, y: x + y, [col(f"user_feat_{i}") * col(f"feature_{i}") for i in range(vec_len)])
user_norm = sqrt(reduce(lambda x, y: x + y, [col(f"user_feat_{i}")**2 for i in range(vec_len)]))
news_norm = sqrt(reduce(lambda x, y: x + y, [col(f"feature_{i}")**2 for i in range(vec_len)]))
user_news = user_news.withColumn("dot", dot_product)\
                     .withColumn("user_norm", user_norm)\
                     .withColumn("news_norm", news_norm)\
                     .withColumn("similarity", col("dot") / (col("user_norm") * col("news_norm")))

In [78]:
windowSpec = Window.partitionBy("User ID").orderBy(col("similarity").desc())
top_recs = user_news.withColumn("rank", row_number().over(windowSpec))\
            .filter(col("rank") <= 5)\
            .select("User ID", "News ID", "similarity", "rank")

In [79]:
# 过滤掉用户已看过的新闻
from pyspark.sql.functions import explode

user_seen = behaviors_df_2.select(
    col("User ID"),
    explode(col("HistoryList")).alias("News ID")
)
top_recs = top_recs.join(user_seen, on = ["User ID", "News ID"], how = "left_anti")

In [80]:
top_recs = top_recs.orderBy("User ID", "rank") \
                   .select("User ID", "News ID", "similarity", "rank") \
                   .toPandas()
top_recs.head(20)

,User ID,News ID,similarity,rank
0,2,57809,0.713097,1
1,2,20079,0.672170,2
2,2,28213,0.577620,3
3,2,33198,0.549661,4
4,2,58964,0.543932,5
5,6,2461,0.591962,1
6,6,8555,0.564557,2
7,6,7210,0.564303,3
8,6,42457,0.562344,4
9,6,62745,0.542843,5


In [81]:
top_recs.to_csv("contentbased_results.csv", index = False)

#### Evaluation（未用到新数据集来eval）

In [26]:
import pandas as pd

true_clicks = behaviors_df_2.select(
    col("User ID"),
    expr("explode(ClickedNews)").alias("Clicked_News_ID")
).toPandas()

predictions = top_recs[['User ID', 'News ID']].rename(columns={"News ID": "Predicted_News_ID"})

merged = pd.merge(predictions, true_clicks, on="User ID", how="inner")
merged["hit"] = (merged["Predicted_News_ID"] == merged["Clicked_News_ID"]).astype(int)

precision_user = merged.groupby("User ID").agg(
    Precision_at_5=("hit", lambda x: x.sum() / 5)
).reset_index()

recall_user = merged.groupby("User ID").agg(
    Recall_at_5=("hit", lambda x: x.sum() / x.count())
).reset_index()

avg_precision = precision_user["Precision_at_5"].mean()
avg_recall = recall_user["Recall_at_5"].mean()

print(f"Precision@5：{avg_precision:.4f}")
print(f"Recall@5：{avg_recall:.4f}")

Precision@5：0.3600
Recall@5：0.1617


### GRU4Rec

In [ ]:
## GRU4Rec

import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np

df = behaviors_df_1.toPandas()

user_encoder = LabelEncoder()
item_encoder = LabelEncoder()
df["user_idx"] = user_encoder.fit_transform(df["User ID"])
df["item_idx"] = item_encoder.fit_transform(df["News_ID"])
df = df.sort_values(by=["user_idx", "Time"])
sessions = df.groupby("user_idx")["item_idx"].apply(list).reset_index(name="item_seq")

class GRUDataset(Dataset):
    def __init__(self, sessions, seq_len=3):
        self.samples = []
        for _, row in sessions.iterrows():
            seq = row["item_seq"]
            for i in range(1, len(seq)):
                hist = seq[max(0, i - seq_len):i]
                target = seq[i]
                self.samples.append((hist, target))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        hist, target = self.samples[idx]
        hist_tensor = torch.tensor(hist, dtype=torch.long)
        target_tensor = torch.tensor(target, dtype=torch.long)
        return hist_tensor, target_tensor

class GRU4Rec(nn.Module):
    def __init__(self, num_items, embedding_dim=16, hidden_dim=32):
        super(GRU4Rec, self).__init__()
        self.embedding = nn.Embedding(num_items, embedding_dim)
        self.gru = nn.GRU(embedding_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, num_items)

    def forward(self, x):
        x = self.embedding(x)
        _, h = self.gru(x)
        out = self.fc(h.squeeze(0))
        return out

device = torch.device("cpu")
dataset = GRUDataset(sessions)
dataloader = DataLoader(dataset, batch_size=64, shuffle=True, collate_fn=lambda x: (
    nn.utils.rnn.pad_sequence([s[0] for s in x], batch_first=True),
    torch.stack([s[1] for s in x])
))

model = GRU4Rec(num_items=len(item_encoder.classes_)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

for epoch in range(5):
    total_loss = 0
    model.train()
    for batch_x, batch_y in dataloader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        optimizer.zero_grad()
        output = model(batch_x)
        loss = criterion(output, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")
    

import numpy as np

def recommend_top_k_gru(model, user_history_dict, item_encoder, k=10, seq_len=3):
    model.eval()  # 設置為評估模式
    recommendations = []

    known_items = set(item_encoder.classes_)
    for user_id, history in user_history_dict.items():
        valid_hist = [str(i) for i in history[-seq_len:] if str(i) in known_items]
        if not valid_hist:
            continue

        hist_encoded = item_encoder.transform(valid_hist)
        hist_tensor = torch.tensor(hist_encoded, dtype=torch.long).unsqueeze(0)

        with torch.no_grad():
            scores = model(hist_tensor)  # (1, num_items)
            topk_scores, topk_indices = torch.topk(scores, k, dim=1)

        topk_items = item_encoder.inverse_transform(topk_indices[0].cpu().tolist())
        user_recs = list(zip(map(int, topk_items), topk_scores[0].cpu().tolist()))
        recommendations.append({
            "user": user_encoder.inverse_transform([user_id])[0],  # 將用戶ID還原
            "recommendations": user_recs
        })

    return recommendations

user_history_dict = {row['user_idx']: row['item_seq'] for _, row in sessions.iterrows()}
topk_recs = recommend_top_k_gru(model, user_history_dict, item_encoder, k=10)
recommendation_df = pd.DataFrame(topk_recs)
print(recommendation_df.head())

### User-based using common clicks

In [17]:
from pyspark.sql.functions import explode, col, count
from pyspark.sql import Window
from pyspark.sql.functions import row_number

# 1. Explode ClickedNews
user_clicks = behaviors_df_2.select(
    col("User ID"),
    explode(col("ClickedNews")).alias("News ID")
).dropna()

# 2. Create user-user co-click table
user_clicks_alias = user_clicks.alias("a")
user_clicks_other = user_clicks.alias("b")

user_pairs = user_clicks_alias.join(
    user_clicks_other,
    on="News ID"
).filter(col("a.User ID") != col("b.User ID")) \
 .select(
     col("a.User ID").alias("User_A"),
     col("b.User ID").alias("User_B"),
     col("News ID")
)

# 3. Count number of shared clicks
user_similarity = user_pairs.groupBy("User_A", "User_B")\
                             .agg(count("News ID").alias("common_clicks"))

# 4. Pick top 5 similar users
windowSpec = Window.partitionBy("User_A").orderBy(col("common_clicks").desc())

top_similar_users = user_similarity.withColumn("rank", row_number().over(windowSpec))\
                                   .filter(col("rank") <= 5)

# 5. Recommend news from similar users
similar_users_clicks = top_similar_users.join(
    user_clicks.withColumnRenamed("User ID", "User_B"),
    on="User_B"
).select(
    col("User_A").alias("User ID"),
    "News ID"
)

# 6. Remove already seen news
user_seen = behaviors_df_2.select(
    col("User ID"),
    explode(col("HistoryList")).alias("News ID")
)

final_recommendations = similar_users_clicks.join(user_seen, on=["User ID", "News ID"], how="left_anti")

# 7. Rank news
windowSpec2 = Window.partitionBy("User ID").orderBy("News ID")

final_recommendations = final_recommendations.withColumn("rank", row_number().over(windowSpec2))\
                                             .filter(col("rank") <= 5)

# 8. View
#final_recommendations.show(20)

# Only to Pandas if safe
final_recommendations_pd = final_recommendations.toPandas()
final_recommendations_pd.head(20)

,User ID,News ID,rank
0,6,1019,1
1,6,10754,2
2,6,12446,3
3,6,12714,4
4,6,13259,5
5,8,12446,1
6,8,14726,2
7,8,15812,3
8,8,16209,4
9,8,16439,5


In [19]:
final_recommendations_pd.to_csv('userbased_final_recommendations.csv', index = False)

#### evaluation using training data

In [18]:
from pyspark.sql.functions import explode, col, collect_list

# Get clicked news from behaviors_small
test_clicks = behaviors_df_2.select(
    col("User ID"),
    explode(col("ClickedNews")).alias("News ID")
).dropna()

# Aggregate clicked news per user
true_news = test_clicks.groupBy("User ID").agg(collect_list("News ID").alias("true_list"))

# Aggregate recommended news per user
rec_news = final_recommendations.groupBy("User ID")\
    .agg(collect_list("News ID").alias("rec_list"))
# Join recommendations with ground truth
eval_df = rec_news.join(true_news, on="User ID", how="inner")

from pyspark.sql.functions import array_intersect, size, lit, avg

# Calculate how many recommended news matched clicked news
eval_df = eval_df.withColumn("num_correct", size(array_intersect(col("rec_list"), col("true_list"))))

# Precision@5 = correct / 5
eval_df = eval_df.withColumn("precision_at_5", col("num_correct") / lit(5))

# Recall@5 = correct / total clicked
eval_df = eval_df.withColumn("recall_at_5", col("num_correct") / size(col("true_list")))

# Average Precision and Recall over all users
precision_at_5 = eval_df.select(avg("precision_at_5")).first()[0]
recall_at_5 = eval_df.select(avg("recall_at_5")).first()[0]

print(f"Precision@5 = {precision_at_5:.4f}")
print(f"Recall@5 = {recall_at_5:.4f}")

Precision@5 = 0.0303
Recall@5 = 0.0567


### Evaluation Function (AUC, MRR, nDCG@K)

In [85]:
from pyspark.sql.functions import col, explode, when, lit, rank
from pyspark.sql.window import Window
from pyspark.sql import functions as F
import math

def evaluate_auc_mrr_ndcg(recommendations_pd, behaviors_val_spark, k=5):
    # Convert pandas recommendations to Spark DataFrame
    recommendations_pd.columns = [c.lower().replace(" ", "_") for c in recommendations_pd.columns]
    recommendations_pd["user_id"] = recommendations_pd["user_id"].astype(str)
    recommendations_pd["news_id"] = recommendations_pd["news_id"].astype(str)
    recommendations_pd["rank"] = recommendations_pd["rank"].astype(int)
    rec_spark = spark.createDataFrame(recommendations_pd)
    
    # Prepare ground truth clicks
    true_clicks = behaviors_val_spark.select(
        col("User ID").alias("user_id"),
        explode("ClickedNews").alias("clicked_news_id")
    )
    
    # Prepare predictions
    preds = rec_spark.select(
        col("user_id"),
        col("news_id").alias("predicted_news_id"),
        col("rank")
    )
    
    # Join predictions with ground truth
    joined = preds.join(true_clicks, on="user_id", how="left") \
                  .withColumn("is_hit", (col("predicted_news_id") == col("clicked_news_id")).cast("int"))

    # Calculate MRR@K
    windowSpec = Window.partitionBy("user_id").orderBy(col("rank"))
    ranked = joined.withColumn("rr", when(col("is_hit") == 1, 1 / col("rank")).otherwise(0))
    mrr = ranked.groupBy("user_id").agg(F.max("rr").alias("rr")).agg(F.mean("rr")).first()[0]
    
    # Calculate nDCG@K
    ranked = ranked.withColumn("dcg_component", when(col("is_hit") == 1, 1 / F.log2(col("rank") + 1)).otherwise(0))
    dcg = ranked.groupBy("user_id").agg(F.sum("dcg_component").alias("dcg"))
    idcg_value = sum([1 / math.log2(i + 1) for i in range(1, k + 1)])
    idcg = lit(idcg_value)
    ndcg = dcg.withColumn("ndcg", col("dcg") / idcg).agg(F.mean("ndcg")).first()[0]

    # Calculate AUC@K
    if joined.filter(col("is_hit") == 1).count() == 0 or joined.filter(col("is_hit") == 0).count() == 0:
        auc = None
    else:
        positive = joined.filter(col("is_hit") == 1).groupBy("user_id").agg(F.count("*").alias("pos"))
        negative = joined.filter(col("is_hit") == 0).groupBy("user_id").agg(F.count("*").alias("neg"))
        auc_df = positive.join(negative, on="user_id", how="inner") \
                     .withColumn("auc", col("pos") / (col("pos") + col("neg")))
        auc = auc_df.agg(F.mean("auc")).first()[0]

    return {
        "MRR@{}".format(k): mrr,
        "nDCG@{}".format(k): ndcg,
        "AUC@{}".format(k): auc
    }

In [53]:
import pandas as pd
final_recommendations_pd = pd.read_csv('userbased_final_recommendations.csv')

In [13]:
metrics = evaluate_auc_mrr_ndcg(final_recommendations_pd, behaviors_val, k = 5)
print(metrics)

25/04/27 17:08:05 WARN TaskSetManager: Stage 11 contains a task of very large size (1305 KiB). The maximum recommended task size is 1000 KiB.
25/04/27 17:08:19 WARN TaskSetManager: Stage 18 contains a task of very large size (1305 KiB). The maximum recommended task size is 1000 KiB.
25/04/27 17:08:36 WARN TaskSetManager: Stage 26 contains a task of very large size (1305 KiB). The maximum recommended task size is 1000 KiB.
25/04/27 17:08:39 WARN TaskSetManager: Stage 29 contains a task of very large size (1305 KiB). The maximum recommended task size is 1000 KiB.


{'MRR@5': 1.570593149540518e-05, 'nDCG@5': 8.959469273019674e-06, 'AUC@5': 0.024572649572649572}


In [86]:
import pandas as pd
contentbased_pd = pd.read_csv('contentbased_results.csv')

In [87]:
metrics_contentbased = evaluate_auc_mrr_ndcg(contentbased_pd, behaviors_val, k = 5)
print(metrics_contentbased)

25/04/29 19:10:28 WARN TaskSetManager: Stage 336 contains a task of very large size (3622 KiB). The maximum recommended task size is 1000 KiB.
25/04/29 19:10:38 WARN TaskSetManager: Stage 343 contains a task of very large size (3622 KiB). The maximum recommended task size is 1000 KiB.
25/04/29 19:10:53 WARN TaskSetManager: Stage 350 contains a task of very large size (3622 KiB). The maximum recommended task size is 1000 KiB.
25/04/29 19:10:59 WARN TaskSetManager: Stage 354 contains a task of very large size (3622 KiB). The maximum recommended task size is 1000 KiB.
25/04/29 19:11:05 WARN TaskSetManager: Stage 359 contains a task of very large size (3622 KiB). The maximum recommended task size is 1000 KiB.
25/04/29 19:11:06 WARN TaskSetManager: Stage 360 contains a task of very large size (3622 KiB). The maximum recommended task size is 1000 KiB.


{'MRR@5': 6.333333333333333e-05, 'nDCG@5': 2.7968592794034008e-05, 'AUC@5': 0.12}


### User-based (jaccard similarity)

In [30]:
from pyspark.sql.functions import col, explode, count, lit
from pyspark.sql import Window
from pyspark.sql.functions import row_number

# 1. Explode ClickedNews
user_clicks = behaviors_df_2.select(
    col("User ID"),
    explode(col("ClickedNews")).alias("News ID")
).dropna()

# 2. Create user-user co-click table
user_clicks_alias = user_clicks.alias("a")
user_clicks_other = user_clicks.alias("b")

user_pairs = user_clicks_alias.join(
    user_clicks_other,
    on="News ID"
).filter(col("a.User ID") != col("b.User ID")) \
.select(
    col("a.User ID").alias("User_A"),
    col("b.User ID").alias("User_B"),
    col("News ID")
)

# 3. Count number of shared clicks (intersection size)
user_common_clicks = user_pairs.groupBy("User_A", "User_B")\
                               .agg(count("News ID").alias("common_clicks"))

# 4. Count total clicks per user
user_total_clicks = user_clicks.groupBy("User ID")\
                               .agg(count("News ID").alias("total_clicks"))

# 5. Join total clicks properly
user_similarity = user_common_clicks \
    .join(user_total_clicks.withColumnRenamed("User ID", "User_A").withColumnRenamed("total_clicks", "total_clicks_A"), on="User_A") \
    .join(user_total_clicks.withColumnRenamed("User ID", "User_B").withColumnRenamed("total_clicks", "total_clicks_B"), on="User_B")

# 6. Calculate Jaccard similarity
user_similarity = user_similarity.withColumn(
    "jaccard_similarity",
    col("common_clicks") / (col("total_clicks_A") + col("total_clicks_B") - col("common_clicks"))
)

# 7. Pick top 5 similar users by Jaccard
windowSpec = Window.partitionBy("User_A").orderBy(col("jaccard_similarity").desc())

top_similar_users = user_similarity.withColumn("rank", row_number().over(windowSpec))\
                                   .filter(col("rank") <= 5)

# 8. Recommend news from top similar users
similar_users_clicks = top_similar_users.join(
    user_clicks.withColumnRenamed("User ID", "User_B"),
    on="User_B"
).select(
    col("User_A").alias("User ID"),
    "News ID",
    "jaccard_similarity"
)

# NEW: Deduplicate User ID + News ID here!
similar_users_clicks = similar_users_clicks.dropDuplicates(["User ID", "News ID"])

# 9. Remove already seen news
user_seen = behaviors_df_2.select(
    col("User ID"),
    explode(col("HistoryList")).alias("News ID")
)

final_recommendations = similar_users_clicks.join(user_seen, on=["User ID", "News ID"], how="left_anti")

# 10. Rank news based on Jaccard similarity
windowSpec2 = Window.partitionBy("User ID").orderBy(col("jaccard_similarity").desc())

userbased_final_recommendations = final_recommendations.withColumn(
    "rank", row_number().over(windowSpec2)
).filter(
    col("rank") <= 5
)

userbased_final_recommendations_pd = userbased_final_recommendations.toPandas()
userbased_final_recommendations_pd.head(20)

,User ID,News ID,jaccard_similarity,rank
0,6,27581,0.166667,1
1,6,35729,0.166667,2
2,6,3123,0.166667,3
3,6,33619,0.166667,4
4,6,33619,0.166667,5
5,8,57402,0.500000,1
6,8,57402,0.500000,2
7,8,57402,0.500000,3
8,8,16439,0.500000,4
9,8,16439,0.500000,5


In [31]:
userbased_final_recommendations_pd.to_csv('userbased_final_rec_jaccard.csv', index = False)

In [32]:
metrics = evaluate_auc_mrr_ndcg(userbased_final_recommendations_pd, behaviors_val, k = 5)
print(metrics)

25/04/27 17:33:21 WARN TaskSetManager: Stage 79 contains a task of very large size (3117 KiB). The maximum recommended task size is 1000 KiB.
25/04/27 17:33:35 WARN TaskSetManager: Stage 86 contains a task of very large size (3117 KiB). The maximum recommended task size is 1000 KiB.
25/04/27 17:33:51 WARN TaskSetManager: Stage 94 contains a task of very large size (3117 KiB). The maximum recommended task size is 1000 KiB.
25/04/27 17:33:54 WARN TaskSetManager: Stage 97 contains a task of very large size (3117 KiB). The maximum recommended task size is 1000 KiB.


{'MRR@5': 8.223684210526317e-05, 'nDCG@5': 4.925188563975436e-05, 'AUC@5': 0.11048245614035088}


### user-based with history clicks (jaccard similarity)

In [36]:
from pyspark.sql.functions import col, explode, count, array_union
from pyspark.sql import Window
from pyspark.sql.functions import row_number

# Step 1: Add ClickedNews into HistoryList
behaviors_df_2 = behaviors_df_2.withColumn(
    "AllClickedNews",
    array_union(col("HistoryList"), col("ClickedNews"))
)

# Step 2: Explode AllClickedNews
user_clicks = behaviors_df_2.select(
    col("User ID"),
    explode(col("AllClickedNews")).alias("News ID")
).dropna()

# 3. Create user-user co-click table
user_clicks_alias = user_clicks.alias("a")
user_clicks_other = user_clicks.alias("b")

user_pairs = user_clicks_alias.join(
    user_clicks_other,
    on="News ID"
).filter(col("a.User ID") != col("b.User ID")) \
.select(
    col("a.User ID").alias("User_A"),
    col("b.User ID").alias("User_B"),
    col("News ID")
)

# 4. Count number of shared clicks (intersection size)
user_common_clicks = user_pairs.groupBy("User_A", "User_B")\
                               .agg(count("News ID").alias("common_clicks"))

# 5. Count total clicks per user
user_total_clicks = user_clicks.groupBy("User ID")\
                               .agg(count("News ID").alias("total_clicks"))

# 6. Join total clicks properly
user_similarity = user_common_clicks \
    .join(user_total_clicks.withColumnRenamed("User ID", "User_A").withColumnRenamed("total_clicks", "total_clicks_A"), on="User_A") \
    .join(user_total_clicks.withColumnRenamed("User ID", "User_B").withColumnRenamed("total_clicks", "total_clicks_B"), on="User_B")

# 7. Calculate Jaccard similarity
user_similarity = user_similarity.withColumn(
    "jaccard_similarity",
    col("common_clicks") / (col("total_clicks_A") + col("total_clicks_B") - col("common_clicks"))
)

# 8. Pick top 5 similar users by Jaccard
windowSpec = Window.partitionBy("User_A").orderBy(col("jaccard_similarity").desc())

top_similar_users = user_similarity.withColumn("rank", row_number().over(windowSpec))\
                                   .filter(col("rank") <= 5)

# 9. Recommend news from top similar users
similar_users_clicks = top_similar_users.join(
    user_clicks.withColumnRenamed("User ID", "User_B"),
    on="User_B"
).select(
    col("User_A").alias("User ID"),
    "News ID",
    "jaccard_similarity"
)

# NEW: Deduplicate User ID + News ID here!
similar_users_clicks = similar_users_clicks.dropDuplicates(["User ID", "News ID"])

# 10. Remove already seen news
user_seen = behaviors_df_2.select(
    col("User ID"),
    explode(col("HistoryList")).alias("News ID")
)

final_recommendations = similar_users_clicks.join(user_seen, on=["User ID", "News ID"], how="left_anti")

# 11. Rank news based on Jaccard similarity
windowSpec2 = Window.partitionBy("User ID").orderBy(col("jaccard_similarity").desc())

final_recommendations = final_recommendations.withColumn(
    "rank", row_number().over(windowSpec2)
).filter(
    col("rank") <= 5
)

# 12. To pandas
userbased_final_rec_pd = final_recommendations.toPandas()
userbased_final_rec_pd.head(20)


,User ID,News ID,jaccard_similarity,rank
0,6,14629,6.859589,1
1,6,1825,6.859589,2
2,6,1864,6.859589,3
3,6,22397,6.859589,4
4,6,26319,6.859589,5
5,8,9120,0.909091,1
6,8,24109,0.909091,2
7,8,35729,0.909091,3
8,8,42634,0.909091,4
9,8,63970,0.909091,5


In [37]:
userbased_final_rec_pd.to_csv('userbased_final_rec_history_jaccard.csv', index = False)

In [38]:
metrics = evaluate_auc_mrr_ndcg(userbased_final_rec_pd, behaviors_val, k = 5)
print(metrics)

25/04/27 19:08:35 WARN TaskSetManager: Stage 186 contains a task of very large size (3067 KiB). The maximum recommended task size is 1000 KiB.
25/04/27 19:08:49 WARN TaskSetManager: Stage 193 contains a task of very large size (3067 KiB). The maximum recommended task size is 1000 KiB.
25/04/27 19:09:06 WARN TaskSetManager: Stage 201 contains a task of very large size (3067 KiB). The maximum recommended task size is 1000 KiB.
25/04/27 19:09:08 WARN TaskSetManager: Stage 204 contains a task of very large size (3067 KiB). The maximum recommended task size is 1000 KiB.


{'MRR@5': 3.2242518038840355e-05, 'nDCG@5': 1.3334324185006835e-05, 'AUC@5': 0.12222222222222223}
